In [1]:
# Model1: XGBoost model to predict Total Alkalinity (TA)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

import optuna  # pip install optuna
import xgboost as xgb  # pip install xgboost


/Users/ethan/Library/Python/3.10/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load TA training dataset (features + target already joined)

data_path = "../../New Datasets/Combined/Final Datasets/ta_training_complete.csv"
full = pd.read_csv(data_path)

print("TA Training dataset shape:", full.shape)
print("Columns:", list(full.columns))
full.head()


TA Training dataset shape: (9319, 24)
Columns: ['latitude', 'longitude', 'month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count', 'total_alkalinity']


,latitude,longitude,month_fitted,swir16,swir22,red,NDMI,MNDWI,pet,aet,...,srad,tmax,tmin,vap,vpd,ws,pdsi,esa_lccs_class,esa_change_count,total_alkalinity
0,-34.405833,19.600556,127.924326,13580.000000,10717.000000,9095.500000,0.166424,-0.184320,132.300003,11.900001,...,263.801483,23.480000,12.179999,1.372,0.79,3.48,-2.59,120.0,0.0,54.181
1,-34.405833,19.600556,130.915586,13929.063497,11630.326299,10277.402037,0.020388,-0.164582,70.400002,67.599998,...,158.596588,17.779999,7.240000,1.004,0.53,3.36,-3.30,120.0,0.0,36.247
2,-34.405833,19.600556,116.838708,14115.066596,11794.309348,10413.610383,0.009226,-0.173084,163.000000,18.000000,...,323.498383,26.469999,15.740000,1.703,0.93,2.07,-3.49,120.0,0.0,57.800
3,-34.405833,19.600556,128.312603,11536.500000,9401.000000,8631.000000,0.129337,-0.139379,51.400002,43.700001,...,113.603378,17.420000,8.400000,1.106,0.45,3.50,-1.60,120.0,0.0,29.719
4,-34.405833,19.600556,130.776727,13263.500000,10589.000000,9197.500000,0.139460,-0.176520,88.200005,58.600002,...,195.097321,19.730000,10.200000,1.231,0.55,3.64,-1.40,120.0,0.0,58.231


In [3]:
# Build feature matrix X and target y for TA

# Columns to exclude from features
exclude_cols = {
    "total_alkalinity",                 # target
    "latitude", "longitude",            # spatial identifiers
}

base_feature_cols = [c for c in full.columns if c not in exclude_cols]
X_full = full[base_feature_cols].copy()
y = full["total_alkalinity"]

print("Initial number of features:", len(base_feature_cols))
print("Features:", base_feature_cols)

# 1) Remove multicollinearity: drop one of each highly correlated pair
corr_matrix = X_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_threshold = 0.95
cols_to_drop_mc = [col for col in upper.columns if any(upper[col] > high_corr_threshold)]

X_mc = X_full.drop(columns=cols_to_drop_mc)
feature_cols_mc = list(X_mc.columns)

print(f"Dropped {len(cols_to_drop_mc)} highly correlated features (>|{high_corr_threshold}|)")
print("Remaining features after multicollinearity reduction:", len(feature_cols_mc))

# Expose reduced set for feature selection
X_reduced_mc = X_mc.copy()
feature_cols_reduced_mc = feature_cols_mc

print("Example remaining feature columns:", feature_cols_reduced_mc[:10])


Initial number of features: 21
Features: ['month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count']
Dropped 2 highly correlated features (>|0.95|)
Remaining features after multicollinearity reduction: 19
Example remaining feature columns: ['month_fitted', 'swir16', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt']


In [4]:
# Feature selection via XGBoost feature importance (on multicollinearity-reduced set)

fs_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

fs_model.fit(X_reduced_mc, y)

importances = fs_model.feature_importances_
fi_fs = pd.DataFrame({"feature": feature_cols_reduced_mc, "importance": importances})
fi_fs = fi_fs.sort_values("importance", ascending=False).reset_index(drop=True)
fi_fs["cum_importance"] = fi_fs["importance"].cumsum()

# Keep features that explain up to 90% of total importance, but ensure at least 20 features
importance_cutoff = 0.90
min_features = 20
selected = fi_fs[fi_fs["cum_importance"] <= importance_cutoff]["feature"].tolist()
if len(selected) < min_features:
    selected = fi_fs.head(min_features)["feature"].tolist()

X = X_reduced_mc[selected].copy()
feature_cols = selected

print("Total features after multicollinearity reduction:", len(feature_cols_reduced_mc))
print("Selected features after importance-based selection:", len(feature_cols))
print("Top selected features:", feature_cols[:15])


Total features after multicollinearity reduction: 19
Selected features after importance-based selection: 19
Top selected features: ['soil', 'esa_lccs_class', 'vpd', 'vap', 'esa_change_count', 'MNDWI', 'tmin', 'ppt', 'pdsi', 'NDMI', 'ws', 'swir16', 'pet', 'red', 'tmax']


In [5]:
# Baseline XGBoost model (for quick R² and importances)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print(f"Baseline Train R²: {r2_score(y_train, y_train_pred):.3f}")
print(f"Baseline Test  R²: {r2_score(y_test, y_test_pred):.3f}")

importances_base = model.feature_importances_
fi_base = pd.DataFrame({"feature": feature_cols, "importance": importances_base})
fi_base.sort_values("importance", ascending=False).head(20)


Baseline Train R²: 0.903
Baseline Test  R²: 0.690


,feature,importance
0,soil,0.113759
1,esa_lccs_class,0.099406
2,vpd,0.072176
3,vap,0.059773
6,tmin,0.059667
11,swir16,0.056567
9,NDMI,0.052904
4,esa_change_count,0.049070
13,red,0.044386
8,pdsi,0.044293


In [6]:
# Stratified K-Fold + Optuna hyperparameter tuning for TA model

n_bins = 10
y_strat = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 0.8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0),
        "gamma": trial.suggest_float("gamma", 0.5, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    cv_scores = []
    for train_idx, valid_idx in skf.split(X, y_strat):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        y_val_pred = model.predict(X_val)
        cv_scores.append(r2_score(y_val, y_val_pred))

    return float(np.mean(cv_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV R²:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


[I 2026-02-27 04:11:28,502] A new study created in memory with name: no-name-179385a3-b558-487d-b541-9ea4517cd033
Best trial: 0. Best value: 0.499403:   2%|▎         | 1/40 [00:02<01:45,  2.70s/it]

[I 2026-02-27 04:11:31,208] Trial 0 finished with value: 0.4994031580113519 and parameters: {'n_estimators': 323, 'max_depth': 5, 'learning_rate': 0.012681151281420604, 'subsample': 0.6273752555821094, 'colsample_bytree': 0.5546116197692579, 'min_child_weight': 17.82437236678216, 'gamma': 2.219868143668867, 'reg_alpha': 0.6326093263322038, 'reg_lambda': 4.095809342964779}. Best is trial 0 with value: 0.4994031580113519.


Best trial: 0. Best value: 0.499403:   5%|▌         | 2/40 [00:04<01:22,  2.16s/it]

[I 2026-02-27 04:11:32,997] Trial 1 finished with value: 0.4387361151773149 and parameters: {'n_estimators': 388, 'max_depth': 3, 'learning_rate': 0.01702280116802573, 'subsample': 0.7361133048114895, 'colsample_bytree': 0.5288856159713576, 'min_child_weight': 10.023979390658255, 'gamma': 1.0008811384375886, 'reg_alpha': 0.13715577189843242, 'reg_lambda': 3.4738440344586436}. Best is trial 0 with value: 0.4994031580113519.


Best trial: 2. Best value: 0.576341:   8%|▊         | 3/40 [00:06<01:10,  1.91s/it]

...

[I 2026-02-27 04:13:27,586] Trial 37 finished with value: 0.6471591387272578 and parameters: {'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.06897329254983421, 'subsample': 0.7684331929717092, 'colsample_bytree': 0.5681574663301177, 'min_child_weight': 10.570318470352696, 'gamma': 3.6748286772957446, 'reg_alpha': 0.8143686131626697, 'reg_lambda': 2.9552913146219666}. Best is trial 25 with value: 0.6630705071798001.


Best trial: 25. Best value: 0.663071:  98%|█████████▊| 39/40 [02:04<00:05,  5.23s/it]

[I 2026-02-27 04:13:32,676] Trial 38 finished with value: 0.6509217814387456 and parameters: {'n_estimators': 367, 'max_depth': 5, 'learning_rate': 0.08878799337052301, 'subsample': 0.6716783818406253, 'colsample_bytree': 0.6034586723622503, 'min_child_weight': 12.280690199848337, 'gamma': 3.3277343964930473, 'reg_alpha': 0.47483950292982935, 'reg_lambda': 4.295072758700008}. Best is trial 25 with value: 0.6630705071798001.


Best trial: 25. Best value: 0.663071: 100%|██████████| 40/40 [02:09<00:00,  3.24s/it]

[I 2026-02-27 04:13:38,236] Trial 39 finished with value: 0.6321711131119784 and parameters: {'n_estimators': 383, 'max_depth': 5, 'learning_rate': 0.05414775688449043, 'subsample': 0.7235942583897432, 'colsample_bytree': 0.5388300696095365, 'min_child_weight': 7.781939713479109, 'gamma': 3.971660674708979, 'reg_alpha': 0.859863281558424, 'reg_lambda': 3.8270556749289377}. Best is trial 25 with value: 0.6630705071798001.
Best CV R²: 0.6630705071798001
Best params:
  n_estimators: 378
  max_depth: 5
  learning_rate: 0.09637541262545964
  subsample: 0.7203744010873694
  colsample_bytree: 0.5822054510896402
  min_child_weight: 8.251041939542482
  gamma: 3.560409468485683
  reg_alpha: 0.4539343219146751
  reg_lambda: 4.009730194960578


In [7]:
# Train final TA model with best hyperparameters and report R² + feature importances

best_params = study.best_params.copy()
best_params.update({"random_state": 42, "n_jobs": -1})

final_model = xgb.XGBRegressor(**best_params)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Final TA model Train R²: {r2_train:.3f}")
print(f"Final TA model Test  R²: {r2_test:.3f}")

importances = final_model.feature_importances_
fi_tuned = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_tuned = fi_tuned.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting TA (tuned model):")
print(fi_tuned.head(20).to_string(index=False))

fi_tuned.head(20)


Final TA model Train R²: 0.868
Final TA model Test  R²: 0.658

Top 20 most important features for predicting TA (tuned model):
         feature  importance
            soil    0.135933
  esa_lccs_class    0.103266
             vpd    0.076838
             vap    0.063030
            tmin    0.060980
          swir16    0.048864
esa_change_count    0.048802
            NDMI    0.048567
           MNDWI    0.045058
             def    0.043363
             red    0.042916
             pet    0.041500
    month_fitted    0.040341
              ws    0.036080
             aet    0.034789
            pdsi    0.034302
            tmax    0.034138
             ppt    0.032384
               q    0.028850


,feature,importance
0,soil,0.135933
1,esa_lccs_class,0.103266
2,vpd,0.076838
3,vap,0.063030
6,tmin,0.060980
11,swir16,0.048864
4,esa_change_count,0.048802
9,NDMI,0.048567
5,MNDWI,0.045058
15,def,0.043363


In [8]:
# Create predictions for TA on validation data and update submission file

# Load current submission file (should have EC and DRP predictions already)
submission_path = "../../submission1.csv"
submission = pd.read_csv(submission_path)
print("Current submission shape:", submission.shape)
print("Submission columns:", list(submission.columns))

# VALIDATION CHECK: Ensure submission has exactly 200 rows
if submission.shape[0] != 200:
    print(f"\n⚠️  WARNING: Submission has {submission.shape[0]} rows instead of 200!")
    print("Loading fresh template instead...")
    submission = pd.read_csv("../../submission_template.csv")
    print(f"Loaded template with shape: {submission.shape}")

# Load validation features
val_features = pd.read_csv("../../New Datasets/Combined/Final Datasets/ta_validation.csv")
print("\nValidation dataset shape:", val_features.shape)
print("Validation columns:", list(val_features.columns))

# Build X_val using same TA feature set
X_val = val_features[feature_cols].copy()
print("\nValidation features shape:", X_val.shape)
print(f"Using {len(feature_cols)} features: {feature_cols[:5]}...")

# Predict TA for validation rows
ta_pred = final_model.predict(X_val)
print(f"\nGenerated {len(ta_pred)} TA predictions")
print(f"TA predictions - Min: {ta_pred.min():.2f}, Max: {ta_pred.max():.2f}, Mean: {ta_pred.mean():.2f}")

# CRITICAL: Match predictions to template by LAT/LON/DATE (not row order!)
val_features['TA_prediction'] = ta_pred

# Standardize column names for merge
val_features_std = val_features.rename(columns={
    'latitude': 'Latitude',
    'longitude': 'Longitude',
    'sample_date': 'Sample Date'
})

# Merge predictions with submission by coordinates AND date
submission_with_ta = submission.merge(
    val_features_std[['Latitude', 'Longitude', 'Sample Date', 'TA_prediction']],
    on=['Latitude', 'Longitude', 'Sample Date'],
    how='left'
)

# Update TA column
submission_with_ta['Total Alkalinity'] = submission_with_ta['TA_prediction']
submission_with_ta = submission_with_ta.drop(columns=['TA_prediction'])

# Ensure column order matches original
submission = submission_with_ta[submission.columns]

# VALIDATION CHECK: Ensure no missing predictions
missing_count = submission['Total Alkalinity'].isnull().sum()
if missing_count > 0:
    print(f"\n⚠️  WARNING: {missing_count} locations without TA predictions!")
    print("Missing locations:")
    print(submission[submission['Total Alkalinity'].isnull()][['Latitude', 'Longitude', 'Sample Date']].head(10))
else:
    print("\n✓ All 200 locations have TA predictions")

# Save final submission file
submission.to_csv(submission_path, index=False)
print(f"\nUpdated submission file: {submission_path}")
print("Submission shape:", submission.shape)
print("\n✓ All three water quality parameters predicted!")
print("\nFinal submission preview:")
submission.head()


Current submission shape: (200, 6)
Submission columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

Validation dataset shape: (200, 24)
Validation columns: ['latitude', 'longitude', 'sample_date', 'month_fitted', 'swir16', 'swir22', 'red', 'NDMI', 'MNDWI', 'pet', 'aet', 'def', 'q', 'ppt', 'soil', 'srad', 'tmax', 'tmin', 'vap', 'vpd', 'ws', 'pdsi', 'esa_lccs_class', 'esa_change_count']

Validation features shape: (200, 19)
Using 19 features: ['soil', 'esa_lccs_class', 'vpd', 'vap', 'esa_change_count']...

Generated 200 TA predictions
TA predictions - Min: 1.44, Max: 193.50, Mean: 77.80

✓ All 200 locations have TA predictions

Updated submission file: ../../submission1.csv
Submission shape: (200, 6)

✓ All three water quality parameters predicted!

Final submission preview:


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,81.448242,262.35413,NaN
1,-33.329167,26.077500,16-09-2015,96.677979,485.01056,NaN
2,-32.991639,27.640028,07-05-2015,50.423111,292.60873,NaN
3,-34.096389,24.439167,07-02-2012,73.046989,417.39285,NaN
4,-32.000556,28.581667,01-10-2014,46.272163,185.54149,NaN


In [9]:
df = pd.read_csv("../../submission1.csv")

# Put Longitude first, then Latitude, then keep the rest in the same order
cols = df.columns.tolist()
fixed_first = ["Longitude", "Latitude", "Sample Date"]
rest = [c for c in cols if c not in fixed_first]
df = df[fixed_first + rest]

df.to_csv("../../submission1.csv", index=False)